## 00_random_catalog — 共有大陸マスク済みランダムカタログ生成

**入力**
- `output/data/australia_mask_polygon.csv` — Natural Earth ベース大陸ポリゴン (polygon_id: 0=本土, 1=Tasmania)

**出力**
- `output/data/random_masked.csv` — 共有ランダムカタログ (N_R = 10,390 点, seed = 42)

**前提**
- `gradle :lib:jar` 完了（`lib/build/libs/retail-utils-1.0.jar` が存在すること）

**設計方針**
- N_R = 10 × max(N_C, N_W) = 10 × 1039 = 10,390（Woolworths 店舗数基準で統一）
- seed = 42 で固定（CC / WW / CW がすべて同一カタログを共有 → RR の分散源を統一）
- `Catalog.generateMaskedCatalog()` は rejection sampling で陸上点のみ採用

In [1]:
@file:DependsOn("../../lib/build/libs/retail-utils-1.0.jar")

import retail.Point
import retail.loadPolygons
import retail.isOnContinent
import retail.generateMaskedCatalog
import retail.estimateLandFraction

In [2]:
// --- ポリゴン読み込みと検証 ---
val polygonPath = "./output/data/australia_mask_polygon.csv"
val polygons = loadPolygons(polygonPath)

polygons.forEach { (id, verts) ->
    val name = if (id == 0) "本土" else "Tasmania"
    println("polygon_id=$id ($name): ${verts.size} 頂点")
}

// 陸上率を Monte Carlo で推定
val fLand = estimateLandFraction(polygons)
println()
println("f_land 推定 = ${"%.3f".format(fLand)}  (理論値 ≈ 0.512)")
println("海洋点割合  = ${"%.1f".format((1.0 - fLand) * 100)} %")

polygon_id=0 (本土): 1751 頂点
polygon_id=1 (Tasmania): 193 頂点

f_land 推定 = 0.502  (理論値 ≈ 0.512)
海洋点割合  = 49.8 %


In [3]:
// --- ランダムカタログ生成 ---
// N_R = 10 × max(N_Coles=685, N_Woolworths=1039) = 10,390
// seed = 42 (CC/WW/CW で共有)
val N_R = 10_390
val SEED = 42L

println("ランダムカタログ生成中... (N_R = $N_R, seed = $SEED)")
val randomCatalog: List<Point> = generateMaskedCatalog(
    n        = N_R,
    polygons = polygons,
    seed     = SEED
)
println("生成完了: ${randomCatalog.size} 点")

// 簡易 QC: 緯度・経度の範囲確認
val lats = randomCatalog.map { it.lat }
val lons = randomCatalog.map { it.lon }
println()
println("QC — 緯度: [%.2f, %.2f]".format(lats.min(), lats.max()))
println("QC — 経度: [%.2f, %.2f]".format(lons.min(), lons.max()))

ランダムカタログ生成中... (N_R = 10390, seed = 42)
生成完了: 10390 点

QC — 緯度: [-43.54, -10.97]
QC — 経度: [113.46, 153.56]


In [4]:
// --- CSV 出力 ---
val outPath = "./output/data/random_masked.csv"
val file = java.io.File(outPath)
file.bufferedWriter().use { w ->
    w.appendLine("lat,lon")
    randomCatalog.forEach { p -> w.appendLine("${p.lat},${p.lon}") }
}
println("保存完了: $outPath")
println("行数: ${randomCatalog.size} (+ ヘッダ 1 行)")

保存完了: ./output/data/random_masked.csv
行数: 10390 (+ ヘッダ 1 行)
